# Task 1 — Load, Inspect, and Clean

In [11]:
import pandas as pd
import numpy as nm
import kagglehub
import os


In [12]:
# Download latest version
path = kagglehub.dataset_download("manishkr1754/cardekho-used-car-data")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'cardekho-used-car-data' dataset.
Path to dataset files: /kaggle/input/cardekho-used-car-data


In [13]:
# List files to find the exact filename
print("Files in dataset:", os.listdir(path))

Files in dataset: ['cardekho_dataset.csv']


In [14]:
# Load the dataset into pandas
file_path = os.path.join(path, "cardekho_dataset.csv")
df = pd.read_csv(file_path)

In [16]:
# Check the status
print(df.head())

   Unnamed: 0       car_name    brand     model  vehicle_age  km_driven  \
0           0    Maruti Alto   Maruti      Alto            9     120000   
1           1  Hyundai Grand  Hyundai     Grand            5      20000   
2           2    Hyundai i20  Hyundai       i20           11      60000   
3           3    Maruti Alto   Maruti      Alto            9      37000   
4           4  Ford Ecosport     Ford  Ecosport            6      30000   

  seller_type fuel_type transmission_type  mileage  engine  max_power  seats  \
0  Individual    Petrol            Manual    19.70     796      46.30      5   
1  Individual    Petrol            Manual    18.90    1197      82.00      5   
2  Individual    Petrol            Manual    17.00    1197      80.00      5   
3  Individual    Petrol            Manual    20.92     998      67.10      5   
4      Dealer    Diesel            Manual    22.77    1498      98.59      5   

   selling_price  
0         120000  
1         550000  
2         2

In [18]:
# Print df.shape
print(df.shape)


(15411, 14)


In [19]:
# Print df.info()
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15411 entries, 0 to 15410
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Unnamed: 0         15411 non-null  int64  
 1   car_name           15411 non-null  object 
 2   brand              15411 non-null  object 
 3   model              15411 non-null  object 
 4   vehicle_age        15411 non-null  int64  
 5   km_driven          15411 non-null  int64  
 6   seller_type        15411 non-null  object 
 7   fuel_type          15411 non-null  object 
 8   transmission_type  15411 non-null  object 
 9   mileage            15411 non-null  float64
 10  engine             15411 non-null  int64  
 11  max_power          15411 non-null  float64
 12  seats              15411 non-null  int64  
 13  selling_price      15411 non-null  int64  
dtypes: float64(2), int64(6), object(6)
memory usage: 1.6+ MB
None


In [20]:
# Print df.isnull().sum() * 100 / len(df)
print(df.isnull().sum() * 100 / len(df))

Unnamed: 0           0.0
car_name             0.0
brand                0.0
model                0.0
vehicle_age          0.0
km_driven            0.0
seller_type          0.0
fuel_type            0.0
transmission_type    0.0
mileage              0.0
engine               0.0
max_power            0.0
seats                0.0
selling_price        0.0
dtype: float64


In [22]:
# Drop rows where selling_price is missing.
df = df.dropna(subset=['selling_price'])

# check df.shape
print(df.shape)

(15411, 14)


In [30]:
# For columns mileage, engine, and max_power — strip any unit strings (e.g. "kmpl", "CC", "bhp"),
# convert to numeric using pd.to_numeric(..., errors='coerce'), and fill remaining nulls with each column's median.

columns_to_fix = ['mileage', 'engine', 'max_power']

for col in columns_to_fix:
    # 1. Strip unit strings
    df[col] = df[col].astype(str).str.replace(r'[^\d.]', '', regex=True)

    # 2. Convert to numeric
    df[col] = pd.to_numeric(df[col], errors='coerce')

    # 3. Fill nulls with the median of the column
    df[col] = df[col].fillna(df[col].median())

# Verify the types and check for remaining nulls
print("\n" * 3)
print(df[columns_to_fix].info())
print("\n" * 3)
print(df[columns_to_fix].head())





<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15411 entries, 0 to 15410
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   mileage    15411 non-null  float64
 1   engine     15411 non-null  int64  
 2   max_power  15411 non-null  float64
dtypes: float64(2), int64(1)
memory usage: 361.3 KB
None




   mileage  engine  max_power
0    19.70     796      46.30
1    18.90    1197      82.00
2    17.00    1197      80.00
3    20.92     998      67.10
4    22.77    1498      98.59


In [37]:
# Remove rows where selling_price is 999999999 or below 10000.

# Keep the rows that match the criteria
df = df[(df['selling_price'] != 999999999) & (df['selling_price'] >= 10000)]
print(df.shape)



(15411, 14)


In [39]:
# Drop all duplicate rows.

df = df.drop_duplicates()

print(df.shape)

(15411, 14)


# Task 2 — Encode Categorical Features

In [40]:
# 1. Apply label encoding to transmission_type — map Manual → 0 and Automatic → 1.

# Create the dictionary
transmission_map = {'Manual': 0, 'Automatic': 1}

# Apply the map to the column
df['transmission_type'] = df['transmission_type'].map(transmission_map)

# Verify if the mapping is done
print(df['transmission_type'].value_counts())

transmission_type
0    12225
1     3186
Name: count, dtype: int64


In [ ]:
# 2. One Hot Coding - Columns to encode
categorical_cols = ['fuel_type', 'seller_type']

# Apply one-hot encoding
df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

In [47]:
# Check the new columns after one hot encoding
print(df.columns)

Index(['Unnamed: 0', 'car_name', 'brand', 'model', 'vehicle_age', 'km_driven',
       'transmission_type', 'mileage', 'engine', 'max_power', 'seats',
       'selling_price', 'fuel_type_Diesel', 'fuel_type_Electric',
       'fuel_type_LPG', 'fuel_type_Petrol', 'seller_type_Individual',
       'seller_type_Trustmark Dealer'],
      dtype='object')


In [46]:
# 3. Print the column list of the final DataFrame using df.columns.tolist().
print(df.columns.tolist())


['Unnamed: 0', 'car_name', 'brand', 'model', 'vehicle_age', 'km_driven', 'transmission_type', 'mileage', 'engine', 'max_power', 'seats', 'selling_price', 'fuel_type_Diesel', 'fuel_type_Electric', 'fuel_type_LPG', 'fuel_type_Petrol', 'seller_type_Individual', 'seller_type_Trustmark Dealer']


# In a markdown cell, answer: Why is drop_first=True used in one-hot encoding, and what does a row of all zeros in the new dummy columns represent?

**Answer: **In statistical modeling and machine learning, setting drop_first=True is the standard way to handle categorical data without introducing mathematical redundancy.

The primary reason for dropping the first column is to avoid the Dummy Variable Trap, a scenario where independent variables are highly correlated.

By dropping one column, we create a Baseline or Reference Category. Every other category is then measured as a difference from that baseline.

A row of all zeros represents the Reference Category (or Baseline). This is the specific category that was dropped by drop_first=True.

Since pandas usually drops the first category alphabetically, the "all zeros" row points to that missing label.

# 3. Task 3 — Split and Compute Baseline MAE

In [48]:
# 1. Define X (all columns except selling_price) and y (selling_price).
# Drop any remaining non-numeric columns from X.

X = df.drop(columns=['selling_price'])
y = df['selling_price']

In [49]:
# 2. Split into 80% train and 20% test using train_test_split with random_state=42.
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [50]:
# 3. Compute the baseline MAE:
# Predict the mean of y_train for every row in the test set and calculate the MAE against y_test.
from sklearn.metrics import mean_absolute_error

# Calculate the baseline MAE
baseline_pred = [y_train.mean()] * len(y_test)
baseline_mae = mean_absolute_error(y_test, baseline_pred)

print("Baseline MAE:", baseline_mae)


Baseline MAE: 468748.45887192385


In [51]:
# 4. Print: "Baseline MAE: ₹X" rounded to the nearest rupee.
print("Baseline MAE: ₹", round(baseline_mae))

Baseline MAE: ₹ 468748
